# FN Mangroves NDVI Recovery Check (Before vs 2 Months After vs Months 3-4 After)

This notebook compares NDVI in FN mangroves across three periods:
- `2 months before` event
- `2 months after` event
- `months 3-4 after` event

Key questions:
1. Did NDVI improve from `2 months after` to `months 3-4 after`?
2. To what extent did mangroves recover back toward `2 months before`?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())

ndvi_before_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif'
ndvi_after2_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif'
ndvi_after34_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_months3to4_after_epsg3448_2025-12-30_to_2026-02-28.tif'
fn_mangroves_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'

output_dir = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images'
output_dir.mkdir(parents=True, exist_ok=True)
SAVE_OUTPUTS = False

for p in [ndvi_before_path, ndvi_after2_path, ndvi_after34_path, fn_mangroves_path]:
    print(p.name, 'exists ->', p.exists())
print('Output dir:', output_dir)


In [ ]:
# Load FN mangroves
mangroves = gpd.read_file(fn_mangroves_path).to_crs(3448)
mangroves = mangroves[mangroves.geometry.notnull() & ~mangroves.geometry.is_empty].copy()

# Load rasters and check alignment
with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after2_path) as src_a2, rasterio.open(ndvi_after34_path) as src_a34:
    if not (src_b.crs == src_a2.crs == src_a34.crs):
        raise ValueError('CRS mismatch among NDVI rasters.')
    if not (src_b.transform == src_a2.transform == src_a34.transform):
        raise ValueError('Transform mismatch among NDVI rasters.')
    if not (src_b.shape == src_a2.shape == src_a34.shape):
        raise ValueError('Shape mismatch among NDVI rasters.')

    before = src_b.read(1)
    after2 = src_a2.read(1)
    after34 = src_a34.read(1)

    transform = src_b.transform
    bounds = src_b.bounds
    raster_crs = src_b.crs

    mangrove_mask = geometry_mask(
        [g for g in mangroves.geometry if g is not None and not g.is_empty],
        transform=transform,
        out_shape=src_b.shape,
        invert=True,
    )

    valid_before = np.isfinite(before) & (before >= -1.0) & (before <= 1.0)
    valid_after2 = np.isfinite(after2) & (after2 >= -1.0) & (after2 <= 1.0)
    valid_after34 = np.isfinite(after34) & (after34 >= -1.0) & (after34 <= 1.0)

    if src_b.nodata is not None and np.isfinite(src_b.nodata):
        valid_before &= before != src_b.nodata
    if src_a2.nodata is not None and np.isfinite(src_a2.nodata):
        valid_after2 &= after2 != src_a2.nodata
    if src_a34.nodata is not None and np.isfinite(src_a34.nodata):
        valid_after34 &= after34 != src_a34.nodata

paired3_mask = mangrove_mask & valid_before & valid_after2 & valid_after34
print('Three-period valid mangrove cells:', f'{int(paired3_mask.sum()):,}')


In [ ]:
# Build table for three-period paired mangrove NDVI
b = before[paired3_mask]
a2 = after2[paired3_mask]
a34 = after34[paired3_mask]

delta_after2_minus_before = a2 - b
delta_after34_minus_after2 = a34 - a2
delta_after34_minus_before = a34 - b

df = pd.DataFrame({
    'ndvi_before': b,
    'ndvi_after2': a2,
    'ndvi_after34': a34,
    'delta_after2_minus_before': delta_after2_minus_before,
    'delta_after34_minus_after2': delta_after34_minus_after2,
    'delta_after34_minus_before': delta_after34_minus_before,
})

summary = pd.DataFrame([
    {
        'metric': 'Mean NDVI before',
        'value': float(df['ndvi_before'].mean()),
    },
    {
        'metric': 'Mean NDVI 2 months after',
        'value': float(df['ndvi_after2'].mean()),
    },
    {
        'metric': 'Mean NDVI months 3-4 after',
        'value': float(df['ndvi_after34'].mean()),
    },
    {
        'metric': 'Mean change: after2 - before',
        'value': float(df['delta_after2_minus_before'].mean()),
    },
    {
        'metric': 'Mean change: after34 - after2',
        'value': float(df['delta_after34_minus_after2'].mean()),
    },
    {
        'metric': 'Mean change: after34 - before',
        'value': float(df['delta_after34_minus_before'].mean()),
    },
]).round(4)

display(summary)


In [ ]:
# Recovery indicators
# - Improved from after2 to after34
# - Recovered to at least before
# - Improved but still below before
# - No improvement / further decline

improved_vs_after2 = df['ndvi_after34'] > df['ndvi_after2']
recovered_to_before = df['ndvi_after34'] >= df['ndvi_before']
improved_but_below_before = improved_vs_after2 & (df['ndvi_after34'] < df['ndvi_before'])
no_improvement_vs_after2 = df['ndvi_after34'] <= df['ndvi_after2']

recovery_table = pd.DataFrame([
    {'metric': 'Improved vs 2 months after (%)', 'value_pct': 100 * improved_vs_after2.mean()},
    {'metric': 'Recovered to/beyond pre-event NDVI (%)', 'value_pct': 100 * recovered_to_before.mean()},
    {'metric': 'Improved but still below pre-event (%)', 'value_pct': 100 * improved_but_below_before.mean()},
    {'metric': 'No improvement or further decline vs 2 months after (%)', 'value_pct': 100 * no_improvement_vs_after2.mean()},
    {'metric': 'Still below pre-event NDVI (%)', 'value_pct': 100 * (df['ndvi_after34'] < df['ndvi_before']).mean()},
]).round(3)

display(recovery_table)


In [ ]:
# Distribution comparison across three periods
fig, ax = plt.subplots(figsize=(10.5, 5.5), constrained_layout=True)

low = min(np.percentile(b, 0.5), np.percentile(a2, 0.5), np.percentile(a34, 0.5))
high = max(np.percentile(b, 99.5), np.percentile(a2, 99.5), np.percentile(a34, 99.5))
bins = np.linspace(low, high, 80)

ax.hist(b, bins=bins, alpha=0.45, density=True, label='Before event', color='#2b8cbe')
ax.hist(a2, bins=bins, alpha=0.45, density=True, label='2 months after', color='#d73027')
ax.hist(a34, bins=bins, alpha=0.45, density=True, label='Months 3-4 after', color='#1a9850')

ax.set_title('FN Mangroves NDVI Distributions Across Three Periods')
ax.set_xlabel('NDVI')
ax.set_ylabel('Density')
ax.legend(frameon=True)

if SAVE_OUTPUTS:
    fig.savefig(output_dir / 'ndvi_mangroves_before_after2_after34_hist.png', dpi=300)

plt.show()


In [ ]:
# Map classification for recovery interpretation + hurricane overlays
# Classes for paired3 mangrove cells:
# -1: no improvement / further decline vs after2
#  0: improved vs after2 but still below before
# +1: recovered to/beyond before

from matplotlib.lines import Line2D
from shapely.ops import unary_union

cls = np.full(before.shape, np.nan, dtype='float32')

no_imp_full = paired3_mask & ((after34 - after2) <= 0)
partial_full = paired3_mask & ((after34 - after2) > 0) & (after34 < before)
recovered_full = paired3_mask & (after34 >= before)

cls[no_imp_full] = -1.0
cls[partial_full] = 0.0
cls[recovered_full] = 1.0

n_no = int(np.nansum(cls == -1.0))
n_partial = int(np.nansum(cls == 0.0))
n_rec = int(np.nansum(cls == 1.0))
n_tot = n_no + n_partial + n_rec

# Load hurricane overlays (track + wind thresholds), clipped to Jamaica-focused AOI
def coerce_track_to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs(4326, allow_override=True)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    return gdf

track_dir = ROOT / 'dphil_papers/dphil_paper_3/inputs/hurricane_melissa_track_noaa/al132025_best_track'
line_path = track_dir / 'AL132025_lin.shp'
windswath_path = track_dir / 'AL132025_windswath.shp'
jamaica_boundary_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
TRACK_AOI_BUFFER_KM = 200
JAMAICA_MAP_BUFFER_KM = 60

jamaica = gpd.read_file(jamaica_boundary_path).to_crs(3448)
track_line_full = coerce_track_to_wgs84(gpd.read_file(line_path)).to_crs(3448)
track_windswath_full = coerce_track_to_wgs84(gpd.read_file(windswath_path)).to_crs(3448)

track_aoi_geom = unary_union(jamaica.geometry).buffer(TRACK_AOI_BUFFER_KM * 1000)
track_aoi = gpd.GeoDataFrame(geometry=[track_aoi_geom], crs=3448)

track_line = gpd.overlay(track_line_full, track_aoi, how='intersection')
track_windswath = gpd.overlay(track_windswath_full, track_aoi, how='intersection')
if len(track_line) == 0:
    track_line = track_line_full.copy()
if len(track_windswath) == 0:
    track_windswath = track_windswath_full.copy()

cmap = ListedColormap(['#d73027', '#fdae61', '#1a9850'])
cmap.set_bad(color='white', alpha=1.0)
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
ax.imshow(
    cls,
    cmap=cmap,
    norm=norm,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    origin='upper',
    interpolation='nearest',
)

# Hurricane wind threshold boundaries (34/50/64 kt)
if len(track_windswath) > 0 and 'RADII' in track_windswath.columns:
    sw = track_windswath.copy()
    sw['RADII'] = pd.to_numeric(sw['RADII'], errors='coerce')
    threshold_colors = {34.0: '#9ecae1', 50.0: '#4292c6', 64.0: '#08519c'}
    for thr in sorted(sw['RADII'].dropna().unique()):
        subset = sw[sw['RADII'] == thr]
        color = threshold_colors.get(float(thr), '#6baed6')
        if len(subset) > 0:
            subset.boundary.plot(ax=ax, color=color, linewidth=1.1, alpha=0.9)

# Track + context boundaries
track_line.plot(ax=ax, color='black', linewidth=1.8, alpha=0.95)
jamaica.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.7)
mangroves.boundary.plot(ax=ax, color='black', linewidth=0.12, alpha=0.3)

# Clip view to Jamaica buffered extent
jamaica_buffer_geom = unary_union(jamaica.geometry.tolist()).buffer(JAMAICA_MAP_BUFFER_KM * 1000)
xmin, ymin, xmax, ymax = jamaica_buffer_geom.bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.set_title('FN Mangroves Recovery Map (3-4 months) with Melissa Track + Wind Thresholds')
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='No improvement / further decline vs 2 months after'),
    mpatches.Patch(facecolor='#fdae61', edgecolor='none', label='Improved vs after2, still below pre-event'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='Recovered to/beyond pre-event NDVI'),
    Line2D([0], [0], color='black', lw=1.8, label='Melissa track line'),
]
if len(track_windswath) > 0 and 'RADII' in track_windswath.columns:
    sw_r = pd.to_numeric(track_windswath['RADII'], errors='coerce')
    for thr, col in [(34.0, '#9ecae1'), (50.0, '#4292c6'), (64.0, '#08519c')]:
        if np.any(sw_r == thr):
            legend_handles.append(Line2D([0], [0], color=col, lw=1.3, label=f'Windswath {int(thr)} kt'))

ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.95, fontsize=8)

ax.text(
    0.01,
    0.01,
    f'n={n_tot:,}  no_improve={100*n_no/max(n_tot,1):.1f}%  partial={100*n_partial/max(n_tot,1):.1f}%  recovered={100*n_rec/max(n_tot,1):.1f}%',
    transform=ax.transAxes,
    fontsize=9,
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.88, edgecolor='none'),
)

if SAVE_OUTPUTS:
    fig.savefig(output_dir / 'ndvi_mangroves_recovery_after34_map_with_track_wind.png', dpi=320)

plt.show()


## Notes
- This notebook uses only FN mangroves and three NDVI layers (`before`, `after2`, `after34`) on EPSG:3448.
- Recovery interpretation here is pixel-wise and threshold-free by default (except valid-data filtering).
- You can add stricter thresholds (for example, `>10%`) if you want a noise-robust recovery definition.


## Added at End: Recovery Within 80% and 90% Damage-Distance Zones

This section replicates the hurricane-track distance-threshold concept for mangroves:
- Build 80% and 90% distance thresholds from the **2-month-after damage distribution** (`(after2-before)/before < -0.10`, baseline `before>=0.20`).
- Then evaluate **months 3-4 recovery** inside those zones.


In [ ]:
# Recovery analysis inside 80%/90% damage-distance zones (track-based)
from shapely.ops import unary_union

# Paths for hurricane track
track_dir = ROOT / 'dphil_papers/dphil_paper_3/inputs/hurricane_melissa_track_noaa/al132025_best_track'
line_path = track_dir / 'AL132025_lin.shp'
jamaica_boundary_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

REL_DAMAGE_THRESHOLD = -0.10
REL_BASELINE_MIN = 0.20
TRACK_AOI_BUFFER_KM = 200


def coerce_track_to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs(4326, allow_override=True)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    return gdf

# Build valid masks for threshold derivation (before + after2 only, as in prior damage-threshold logic)
with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after2_path) as src_a2, rasterio.open(ndvi_after34_path) as src_a34:
    b_full = src_b.read(1)
    a2_full = src_a2.read(1)
    a34_full = src_a34.read(1)

    valid_b = np.isfinite(b_full) & (b_full >= -1.0) & (b_full <= 1.0)
    valid_a2 = np.isfinite(a2_full) & (a2_full >= -1.0) & (a2_full <= 1.0)
    valid_a34 = np.isfinite(a34_full) & (a34_full >= -1.0) & (a34_full <= 1.0)

    if src_b.nodata is not None and np.isfinite(src_b.nodata):
        valid_b &= b_full != src_b.nodata
    if src_a2.nodata is not None and np.isfinite(src_a2.nodata):
        valid_a2 &= a2_full != src_a2.nodata
    if src_a34.nodata is not None and np.isfinite(src_a34.nodata):
        valid_a34 &= a34_full != src_a34.nodata

    paired_after2 = mangrove_mask & valid_b & valid_a2
    eligible_after2 = paired_after2 & (b_full >= REL_BASELINE_MIN)

    rel_after2 = np.full(b_full.shape, np.nan, dtype='float32')
    rel_after2[eligible_after2] = (a2_full[eligible_after2] - b_full[eligible_after2]) / b_full[eligible_after2]
    damaged_after2 = eligible_after2 & (rel_after2 < REL_DAMAGE_THRESHOLD)

    # Recovery comparators require after34 valid too
    paired_for_recovery = eligible_after2 & valid_a34

# Track line clipped to Jamaica AOI (same approach as hurricane notebook)
track_line_full = coerce_track_to_wgs84(gpd.read_file(line_path)).to_crs(3448)
jamaica = gpd.read_file(jamaica_boundary_path).to_crs(3448)
track_aoi_geom = unary_union(jamaica.geometry).buffer(TRACK_AOI_BUFFER_KM * 1000)
track_aoi = gpd.GeoDataFrame(geometry=[track_aoi_geom], crs=3448)
track_line = gpd.overlay(track_line_full, track_aoi, how='intersection')
if len(track_line) == 0:
    track_line = track_line_full.copy()
line_union = unary_union(track_line.geometry.tolist())

# Distances for threshold derivation (eligible_after2 pixels)
rows_e, cols_e = np.where(eligible_after2)
xs_e, ys_e = rasterio.transform.xy(transform, rows_e, cols_e, offset='center')
elig_gdf = gpd.GeoDataFrame(
    {
        'rel_after2': rel_after2[eligible_after2],
        'is_damaged_after2': rel_after2[eligible_after2] < REL_DAMAGE_THRESHOLD,
    },
    geometry=gpd.points_from_xy(xs_e, ys_e),
    crs=3448,
)
elig_gdf['distance_km'] = elig_gdf.geometry.distance(line_union) / 1000.0

dmg_dist = elig_gdf.loc[elig_gdf['is_damaged_after2'], 'distance_km'].to_numpy()
d80 = float(np.quantile(dmg_dist, 0.80))
d90 = float(np.quantile(dmg_dist, 0.90))

# Build recovery table on cells that have before+after2+after34 valid and baseline eligible
rows_r, cols_r = np.where(paired_for_recovery)
xs_r, ys_r = rasterio.transform.xy(transform, rows_r, cols_r, offset='center')

rec = gpd.GeoDataFrame(
    {
        'before': b_full[paired_for_recovery],
        'after2': a2_full[paired_for_recovery],
        'after34': a34_full[paired_for_recovery],
        'rel_after2': rel_after2[paired_for_recovery],
    },
    geometry=gpd.points_from_xy(xs_r, ys_r),
    crs=3448,
)
rec['is_damaged_after2'] = rec['rel_after2'] < REL_DAMAGE_THRESHOLD
rec['distance_km'] = rec.geometry.distance(line_union) / 1000.0

rec['drop_after2_vs_before'] = rec['before'] - rec['after2']
rec['recovery_after34_vs_after2'] = rec['after34'] - rec['after2']
rec['gap_after34_vs_before'] = rec['before'] - rec['after34']

# Avoid unstable ratio where initial drop <= 0
valid_ratio = rec['drop_after2_vs_before'] > 0
rec['recovery_fraction_of_drop'] = np.nan
rec.loc[valid_ratio, 'recovery_fraction_of_drop'] = rec.loc[valid_ratio, 'recovery_after34_vs_after2'] / rec.loc[valid_ratio, 'drop_after2_vs_before']

rows_out = []
for label, thr in [('80% damage distance', d80), ('90% damage distance', d90)]:
    zone_all = rec[rec['distance_km'] <= thr]
    zone_dmg = zone_all[zone_all['is_damaged_after2']]

    for subset_name, z in [('All mangroves in zone', zone_all), ('Damaged (>10%) at 2 months after, in zone', zone_dmg)]:
        if len(z) == 0:
            continue
        rows_out.append({
            'zone': label,
            'distance_km': round(thr, 3),
            'subset': subset_name,
            'n_cells': int(len(z)),
            'mean_before': float(z['before'].mean()),
            'mean_after2': float(z['after2'].mean()),
            'mean_after34': float(z['after34'].mean()),
            'mean_drop_after2_vs_before': float((z['after2'] - z['before']).mean()),
            'mean_recovery_after34_vs_after2': float((z['after34'] - z['after2']).mean()),
            'mean_gap_after34_vs_before': float((z['after34'] - z['before']).mean()),
            'pct_improved_after34_vs_after2': float(100.0 * (z['after34'] > z['after2']).mean()),
            'pct_recovered_to_or_above_before': float(100.0 * (z['after34'] >= z['before']).mean()),
            'mean_recovery_fraction_of_drop_pct': float(100.0 * z['recovery_fraction_of_drop'].mean(skipna=True)),
        })

zone_recovery_df = pd.DataFrame(rows_out)
num_cols = [
    'mean_before', 'mean_after2', 'mean_after34',
    'mean_drop_after2_vs_before', 'mean_recovery_after34_vs_after2', 'mean_gap_after34_vs_before',
    'pct_improved_after34_vs_after2', 'pct_recovered_to_or_above_before', 'mean_recovery_fraction_of_drop_pct'
]
zone_recovery_df[num_cols] = zone_recovery_df[num_cols].round(3)

display(zone_recovery_df)

print(f'80% damage-distance threshold: {d80:.3f} km')
print(f'90% damage-distance threshold: {d90:.3f} km')
